In [2]:
pip install pytorch-tabnet 

Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler
import joblib
import warnings
from pytorch_tabnet.tab_model import TabNetClassifier
from pytorch_tabnet.multitask import TabNetMultiTaskClassifier
from tqdm import tqdm
import torch

# Suppress warnings
warnings.filterwarnings("ignore")
torch.cuda.empty_cache()

# 1. Data Loading and Preprocessing
def load_and_preprocess():
    print("Loading and preprocessing data...")
    data = pd.read_csv('full_data_unhealthy_imputed.csv')
    
    # Validate required columns
    required_cols = ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 'hour', 'date', 
                    'oestrus', 'calving', 'lameness', 'mastitis', 'cow']
    missing_cols = set(required_cols) - set(data.columns)
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")
    
    # Feature engineering
    rest_arr = data['REST'].values
    eat_arr = data['EAT'].values
    
    # Calculate ratio safely
    with np.errstate(divide='ignore', invalid='ignore'):
        ratio = np.where(
            (eat_arr == 0) & (rest_arr == 0),
            1.0,
            rest_arr / np.where(eat_arr == 0, np.nan, eat_arr)
        )
    data['rest_to_eat_ratio'] = pd.Series(ratio).fillna(1.0)
    
    # Cyclical time features
    hour_arr = data['hour'].values
    data['hour_sin'] = np.sin(2 * np.pi * hour_arr / 24)
    data['hour_cos'] = np.cos(2 * np.pi * hour_arr / 24)
    
    features = ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 
               'rest_to_eat_ratio', 'hour_sin', 'hour_cos']
    targets = ['oestrus', 'calving', 'lameness', 'mastitis']
    
    # Clean data
    data = data.dropna(subset=features + targets + ['cow'])
    
    return data[features], data[targets], data['cow']

# 2. TabNet Model Training with Class Weighting
def train_tabnet(X, y, groups):
    # Convert to numpy arrays
    X = X.values.astype(np.float32)
    y = y.values.astype(np.float32)
    
    # Calculate class weights for each target
    class_weights = []
    for i in range(y.shape[1]):
        unique, counts = np.unique(y[:, i], return_counts=True)
        weight = counts[0] / counts[1] if len(counts) > 1 else 1.0
        class_weights.append(weight)
    
    # TabNet configuration
    tabnet_params = {
        'n_d': 16,
        'n_a': 16,
        'n_steps': 3,
        'gamma': 1.3,
        'lambda_sparse': 1e-3,
        'optimizer_fn': torch.optim.Adam,
        'optimizer_params': {'lr': 2e-2, 'weight_decay': 1e-5},
        'scheduler_params': {'step_size': 10, 'gamma': 0.9},
        'scheduler_fn': torch.optim.lr_scheduler.StepLR,
        'mask_type': 'entmax',
        'device_name': 'cuda' if torch.cuda.is_available() else 'cpu',
        'verbose': 0
    }
    
    # Create model
    model = TabNetMultiTaskClassifier(
        n_targets=y.shape[1],
        **tabnet_params
    )
    
    # GroupKFold cross-validation
    gkf = GroupKFold(n_splits=3)
    best_models = []
    best_thresholds = []
    all_metrics = []
    
    for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
        print(f"\nTraining Fold {fold+1}/3")
        
        # Split data
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
        
        # Apply class weights through sample weighting
        sample_weights = np.ones(len(y_train))
        for i in range(y.shape[1]):
            pos_mask = (y_train[:, i] == 1)
            sample_weights[pos_mask] *= class_weights[i]
        
        # Train model
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            eval_name=['val'],
            eval_metric=['auc'],
            weights=sample_weights,
            max_epochs=50,
            patience=10,
            batch_size=1024,
            virtual_batch_size=256,
            num_workers=0,
            drop_last=False
        )
        
        # Predict probabilities
        y_proba = model.predict_proba(X_val)
        y_proba = np.array([p[:, 1] for p in y_proba]).T
        
        # Find optimal thresholds
        thresholds = []
        for i in range(y.shape[1]):
            fpr, tpr, thresh = precision_recall_curve(y_val[:, i], y_proba[:, i])
            f1_scores = 2 * (precision * recall) / (precision + recall + 1e-6)
            best_idx = np.argmax(f1_scores)
            thresholds.append(thresh[best_idx])
        
        best_thresholds.append(thresholds)
        
        # Evaluate
        y_pred = (y_proba > np.array(thresholds)).astype(int)
        
        print(f"\nFold {fold+1} Validation Performance:")
        metrics = []
        for i, target in enumerate(['oestrus', 'calving', 'lameness', 'mastitis']):
            report = classification_report(y_val[:, i], y_pred[:, i], output_dict=True)
            auc = roc_auc_score(y_val[:, i], y_proba[:, i])
            print(f"{target} (AUC={auc:.3f}, Threshold={thresholds[i]:.3f}):")
            print(f"  Precision: {report['1']['precision']:.3f}, Recall: {report['1']['recall']:.3f}, F1: {report['1']['f1-score']:.3f}")
            metrics.append({
                'target': target,
                'auc': auc,
                'precision': report['1']['precision'],
                'recall': report['1']['recall'],
                'f1': report['1']['f1-score'],
                'threshold': thresholds[i]
            })
        
        all_metrics.append(metrics)
        best_models.append(model)
    
    # Select best model based on average F1 score
    avg_f1_scores = [
        np.mean([metrics[i]['f1'] for i in range(len(metrics))])
        for metrics in all_metrics
    ]
    best_fold = np.argmax(avg_f1_scores)
    print(f"\nSelected best model from Fold {best_fold+1}")
    
    return best_models[best_fold], best_thresholds[best_fold], all_metrics[best_fold]

# 3. Main Training Function
def train_and_evaluate():
    # Load data
    X, y, groups = load_and_preprocess()
    print(f"Data shape: {X.shape}, Targets shape: {y.shape}")
    
    # Scale features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Train TabNet
    model, thresholds, metrics = train_tabnet(pd.DataFrame(X_scaled, columns=X.columns), 
                                            y, groups)
    
    # Save model
    model_package = {
        'model': model,
        'thresholds': thresholds,
        'scaler': scaler,
        'features': X.columns.tolist(),
        'metrics': metrics
    }
    joblib.dump(model_package, 'tabnet_cattle_model.pkl')
    print("\nTabNet model saved successfully!")
    return model_package

# 4. Prediction Function
def predict(new_data):
    try:
        model_package = joblib.load('tabnet_cattle_model.pkl')
    except FileNotFoundError:
        raise RuntimeError("Model not found. Train the model first.")
    
    # Feature engineering
    X_new = new_data.copy()
    if {'EAT', 'REST'}.issubset(X_new.columns):
        rest_arr = X_new['REST'].values
        eat_arr = X_new['EAT'].values
        with np.errstate(divide='ignore', invalid='ignore'):
            ratio = np.where(
                (eat_arr == 0) & (rest_arr == 0),
                1.0,
                rest_arr / np.where(eat_arr == 0, np.nan, eat_arr)
            )
        X_new['rest_to_eat_ratio'] = pd.Series(ratio).fillna(1.0)
    
    if 'hour' in X_new.columns:
        hour_arr = X_new['hour'].values
        X_new['hour_sin'] = np.sin(2 * np.pi * hour_arr / 24)
        X_new['hour_cos'] = np.cos(2 * np.pi * hour_arr / 24)
    
    # Select and scale features
    features = model_package['features']
    X_new = X_new[features]
    X_scaled = model_package['scaler'].transform(X_new)
    
    # Predict probabilities
    y_proba = model_package['model'].predict_proba(X_scaled)
    y_proba = np.array([p[:, 1] for p in y_proba]).T
    
    # Apply thresholds
    thresholds = model_package['thresholds']
    results = pd.DataFrame({
        'oestrus': (y_proba[:, 0] > thresholds[0]).astype(int),
        'oestrus_prob': y_proba[:, 0],
        'calving': (y_proba[:, 1] > thresholds[1]).astype(int),
        'calving_prob': y_proba[:, 1],
        'lameness': (y_proba[:, 2] > thresholds[2]).astype(int),
        'lameness_prob': y_proba[:, 2],
        'mastitis': (y_proba[:, 3] > thresholds[3]).astype(int),
        'mastitis_prob': y_proba[:, 3]
    })
    
    return results

if __name__ == "__main__":
    try:
        print("Starting cattle health prediction with TabNet...")
        model_package = train_and_evaluate()
        
        # Print final metrics
        print("\nFinal Model Performance:")
        for metric in model_package['metrics']:
            print(f"{metric['target']}: AUC={metric['auc']:.3f}, F1={metric['f1']:.3f}, "
                  f"Precision={metric['precision']:.3f}, Recall={metric['recall']:.3f}")
        
        # Example prediction
        test_data = pd.DataFrame({
            'IN_ALLEYS': [35.6, 0.0, 10.2],
            'REST': [3564.4, 3599.9, 2500.0],
            'EAT': [0.0, 0.0, 500.0],
            'ACTIVITY_LEVEL': [-814.1, -827.9, -700.0],
            'hour': [1, 2, 3]
        })
        
        print("\nExample predictions:")
        print(predict(test_data))
        
    except Exception as e:
        print(f"\nError: {str(e)}")

Starting cattle health prediction with TabNet...
Loading and preprocessing data...
Data shape: (1935667, 7), Targets shape: (1935667, 4)

Error: TabModel.__init__() got an unexpected keyword argument 'n_targets'


In [6]:
pip install tqdm

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve
from sklearn.preprocessing import StandardScaler
import joblib
import warnings
from pytorch_tabnet.tab_model import TabNetClassifier
import torch
from tqdm import tqdm

# Suppress warnings
warnings.filterwarnings("ignore")
torch.cuda.empty_cache()

# 1. Data Loading and Preprocessing
def load_and_preprocess():
    print("Loading and preprocessing data...")
    data = pd.read_csv('full_data_unhealthy_imputed.csv')
    
    # Validate required columns
    required_cols = ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 'hour', 'date', 
                    'oestrus', 'calving', 'lameness', 'mastitis', 'cow']
    missing_cols = set(required_cols) - set(data.columns)
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")
    
    # Feature engineering
    rest_arr = data['REST'].values
    eat_arr = data['EAT'].values
    
    # Calculate ratio safely
    with np.errstate(divide='ignore', invalid='ignore'):
        ratio = np.where(
            (eat_arr == 0) & (rest_arr == 0),
            1.0,
            rest_arr / np.where(eat_arr == 0, np.nan, eat_arr)
        )
    data['rest_to_eat_ratio'] = pd.Series(ratio).fillna(1.0)
    
    # Cyclical time features
    hour_arr = data['hour'].values
    data['hour_sin'] = np.sin(2 * np.pi * hour_arr / 24)
    data['hour_cos'] = np.cos(2 * np.pi * hour_arr / 24)
    
    features = ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 
               'rest_to_eat_ratio', 'hour_sin', 'hour_cos']
    targets = ['oestrus', 'calving', 'lameness', 'mastitis']
    
    # Clean data
    data = data.dropna(subset=features + targets + ['cow'])
    
    return data[features], data[targets], data['cow']

# 2. TabNet Model Training with Class Weighting
def train_tabnet(X, y, groups):
    # Convert to numpy arrays
    X = X.values.astype(np.float32)
    y = y.values.astype(np.float32)
    
    # TabNet configuration
    tabnet_params = {
        'n_d': 16,
        'n_a': 16,
        'n_steps': 3,
        'gamma': 1.3,
        'lambda_sparse': 1e-3,
        'optimizer_fn': torch.optim.Adam,
        'optimizer_params': {'lr': 2e-2, 'weight_decay': 1e-5},
        'scheduler_params': {'step_size': 10, 'gamma': 0.9},
        'scheduler_fn': torch.optim.lr_scheduler.StepLR,
        'mask_type': 'entmax',
        'device_name': 'cuda' if torch.cuda.is_available() else 'cpu',
        'verbose': 0
    }
    
    # GroupKFold cross-validation
    gkf = GroupKFold(n_splits=3)
    best_models = []
    best_thresholds = []
    all_metrics = []
    
    for fold, (train_idx, val_idx) in enumerate(gkf.split(X, groups=groups)):
        print(f"\nTraining Fold {fold+1}/3")
        
        # Split data
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
        
        # Train models for each target separately
        fold_models = []
        fold_thresholds = []
        fold_metrics = []
        
        for target_idx, target_name in enumerate(['oestrus', 'calving', 'lameness', 'mastitis']):
            print(f"\nTraining model for {target_name} ({target_idx+1}/4)")
            
            # Create model
            model = TabNetClassifier(**tabnet_params)
            
            # Get single target
            y_train_target = y_train[:, target_idx]
            y_val_target = y_val[:, target_idx]
            
            # Apply class weights through sample weighting
            class_0 = np.sum(y_train_target == 0)
            class_1 = np.sum(y_train_target == 1)
            sample_weights = np.ones(len(y_train_target))
            
            if class_1 > 0:  # Only apply weighting if positive samples exist
                weight = class_0 / class_1
                sample_weights[y_train_target == 1] = weight
            else:
                print(f"Warning: No positive samples for {target_name} in training set")
            
            # Train model
            model.fit(
                X_train, y_train_target,
                eval_set=[(X_val, y_val_target)],
                eval_name=['val'],
                eval_metric=['logloss'],
                weights=sample_weights,
                max_epochs=30,
                patience=7,
                batch_size=2048,
                virtual_batch_size=512,
                num_workers=0,
                drop_last=False
            )
            
            # Predict probabilities
            y_proba = model.predict_proba(X_val)[:, 1]
            
            # Find optimal threshold
            precision, recall, thresh = precision_recall_curve(y_val_target, y_proba)
            f1_scores = 2 * (precision * recall) / (precision + recall + 1e-6)
            best_idx = np.argmax(f1_scores)
            threshold = thresh[best_idx]
            
            # Evaluate
            y_pred = (y_proba > threshold).astype(int)
            
            # Robust classification report handling
            try:
                report = classification_report(y_val_target, y_pred, output_dict=True, zero_division=0)
                
                # Check if positive class exists in report
                if '1' in report:
                    precision_val = report['1']['precision']
                    recall_val = report['1']['recall']
                    f1_val = report['1']['f1-score']
                else:
                    precision_val = 0.0
                    recall_val = 0.0
                    f1_val = 0.0
            except:
                precision_val = 0.0
                recall_val = 0.0
                f1_val = 0.0
            
            try:
                auc = roc_auc_score(y_val_target, y_proba)
            except:
                auc = 0.5  # Default to random performance
            
            print(f"{target_name} (AUC={auc:.3f}, Threshold={threshold:.3f}):")
            print(f"  Precision: {precision_val:.3f}, Recall: {recall_val:.3f}, F1: {f1_val:.3f}")
            
            fold_models.append(model)
            fold_thresholds.append(threshold)
            fold_metrics.append({
                'target': target_name,
                'auc': auc,
                'precision': precision_val,
                'recall': recall_val,
                'f1': f1_val,
                'threshold': threshold
            })
        
        best_models.append(fold_models)
        best_thresholds.append(fold_thresholds)
        all_metrics.append(fold_metrics)
    
    # Select best fold based on average F1 score
    avg_f1_scores = [
        np.mean([m['f1'] for m in metrics])
        for metrics in all_metrics
    ]
    best_fold = np.argmax(avg_f1_scores)
    print(f"\nSelected best model from Fold {best_fold+1}")
    
    return best_models[best_fold], best_thresholds[best_fold], all_metrics[best_fold]

# 3. Main Training Function
def train_and_evaluate():
    # Load data
    X, y, groups = load_and_preprocess()
    print(f"Data shape: {X.shape}, Targets shape: {y.shape}")
    
    # Scale features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Train TabNet
    models, thresholds, metrics = train_tabnet(pd.DataFrame(X_scaled, columns=X.columns), 
                                             y, groups)
    
    # Save model
    model_package = {
        'models': models,
        'thresholds': thresholds,
        'scaler': scaler,
        'features': X.columns.tolist(),
        'metrics': metrics
    }
    joblib.dump(model_package, 'tabnet_cattle_model.pkl')
    print("\nTabNet model saved successfully!")
    return model_package

# 4. Prediction Function
def predict(new_data):
    try:
        model_package = joblib.load('tabnet_cattle_model.pkl')
    except FileNotFoundError:
        raise RuntimeError("Model not found. Train the model first.")
    
    # Feature engineering
    X_new = new_data.copy()
    if {'EAT', 'REST'}.issubset(X_new.columns):
        rest_arr = X_new['REST'].values
        eat_arr = X_new['EAT'].values
        with np.errstate(divide='ignore', invalid='ignore'):
            ratio = np.where(
                (eat_arr == 0) & (rest_arr == 0),
                1.0,
                rest_arr / np.where(eat_arr == 0, np.nan, eat_arr)
            )
        X_new['rest_to_eat_ratio'] = pd.Series(ratio).fillna(1.0)
    
    if 'hour' in X_new.columns:
        hour_arr = X_new['hour'].values
        X_new['hour_sin'] = np.sin(2 * np.pi * hour_arr / 24)
        X_new['hour_cos'] = np.cos(2 * np.pi * hour_arr / 24)
    
    # Select and scale features
    features = model_package['features']
    X_new = X_new[features]
    X_scaled = model_package['scaler'].transform(X_new)
    
    # Predict probabilities for each target
    y_proba = []
    for i, model in enumerate(model_package['models']):
        proba = model.predict_proba(X_scaled)[:, 1]
        y_proba.append(proba)
    
    # Apply thresholds
    thresholds = model_package['thresholds']
    results = pd.DataFrame({
        'oestrus': (y_proba[0] > thresholds[0]).astype(int),
        'oestrus_prob': y_proba[0],
        'calving': (y_proba[1] > thresholds[1]).astype(int),
        'calving_prob': y_proba[1],
        'lameness': (y_proba[2] > thresholds[2]).astype(int),
        'lameness_prob': y_proba[2],
        'mastitis': (y_proba[3] > thresholds[3]).astype(int),
        'mastitis_prob': y_proba[3]
    })
    
    return results

if __name__ == "__main__":
    try:
        print("Starting cattle health prediction with TabNet...")
        model_package = train_and_evaluate()
        
        # Print final metrics
        print("\nFinal Model Performance:")
        for metric in model_package['metrics']:
            print(f"{metric['target']}: AUC={metric['auc']:.3f}, F1={metric['f1']:.3f}, "
                  f"Precision={metric['precision']:.3f}, Recall={metric['recall']:.3f}")
        
        # Example prediction
        test_data = pd.DataFrame({
            'IN_ALLEYS': [35.6, 0.0, 10.2],
            'REST': [3564.4, 3599.9, 2500.0],
            'EAT': [0.0, 0.0, 500.0],
            'ACTIVITY_LEVEL': [-814.1, -827.9, -700.0],
            'hour': [1, 2, 3]
        })
        
        print("\nExample predictions:")
        print(predict(test_data))
        
    except Exception as e:
        print(f"\nError: {str(e)}")

Starting cattle health prediction with TabNet...
Loading and preprocessing data...
Data shape: (1935667, 7), Targets shape: (1935667, 4)

Training Fold 1/3

Training model for oestrus (1/4)

Early stopping occurred at epoch 9 with best_epoch = 2 and best_val_logloss = 0.60743
oestrus (AUC=0.665, Threshold=0.880):
  Precision: 0.000, Recall: 0.000, F1: 0.000

Training model for calving (2/4)

Early stopping occurred at epoch 12 with best_epoch = 5 and best_val_logloss = 0.30491
calving (AUC=0.337, Threshold=0.862):
  Precision: 0.000, Recall: 0.000, F1: 0.000

Training model for lameness (3/4)

Early stopping occurred at epoch 9 with best_epoch = 2 and best_val_logloss = 0.5945
lameness (AUC=0.587, Threshold=0.794):
  Precision: 0.000, Recall: 0.000, F1: 0.000

Training model for mastitis (4/4)
